# 07 · Turn Arrow summaries into a dashboard

Build a small charting workflow without moving millions of rows into pandas.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [NYC TLC trip records and taxi-zone lookup](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
This lesson uses the January 2024 yellow-taxi Parquet file. See the
[NYC Open Data terms of use](https://opendata.cityofnewyork.us/overview/#termsofuse)
and TLC's data dictionary and accuracy notice; do not treat public availability
as a warranty or apply the notebook's code license to the data.
The original Parquet download is roughly 50 MB and contains roughly three million
rows. The helper checks the pinned file digest. Queries use explicit result
limits; a small result does not imply a small source scan.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


In [ ]:
taxi = lab.taxi()
config = lab.catalog(sources=[{"id": "trips", "type": "parquet", "path": str(taxi)}])
daily_result = lab.query(config, """
    SELECT CAST(tpep_pickup_datetime AS DATE) AS day,
           COUNT(*)::BIGINT AS trips,
           AVG(trip_distance) FILTER (WHERE trip_distance > 0 AND trip_distance < 100) AS mean_distance
    FROM trips
    WHERE tpep_pickup_datetime >= TIMESTAMP '2024-01-01'
      AND tpep_pickup_datetime < TIMESTAMP '2024-02-01'
    GROUP BY 1 ORDER BY day
""", sources=["trips"], max_rows=40)
payments_result = lab.query(config, """
    SELECT CASE payment_type WHEN 1 THEN 'Card' WHEN 2 THEN 'Cash' ELSE 'Other / unknown' END AS payment,
           COUNT(*)::BIGINT AS trips
    FROM trips
    WHERE tpep_pickup_datetime >= TIMESTAMP '2024-01-01'
      AND tpep_pickup_datetime < TIMESTAMP '2024-02-01'
    GROUP BY 1 ORDER BY trips DESC, payment
""", sources=["trips"], max_rows=10)
daily, payments = daily_result.table().to_pandas(), payments_result.table().to_pandas()
assert len(daily) == 31
assert int(daily.trips.sum()) == int(payments.trips.sum())
assert all(daily.mean_distance > 0)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
plt.rcParams.update({"figure.facecolor": "#faf8f4", "axes.facecolor": "#faf8f4",
                     "axes.edgecolor": "#333333", "font.size": 11})
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
axes[0].plot(daily.day, daily.trips, color="#d42d32", linewidth=2)
axes[0].set(title="Daily yellow-taxi trips", ylabel="Trips", xlabel="January 2024")
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1000:.0f}k"))
axes[0].tick_params(axis="x", rotation=35)
axes[0].grid(axis="y", color="#dddddd", linewidth=0.7)
axes[1].barh(payments.payment, payments.trips, color=["#d42d32", "#242424", "#929292"][:len(payments)])
axes[1].invert_yaxis()
axes[1].set(title="Recorded payment methods", xlabel="Trips")
axes[1].xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1e6:.1f}M"))
fig.suptitle("NYC demand overview · powered by Kelvo", fontsize=16)
plt.show()
artifact = lab.root / "taxi-dashboard.png"
fig.savefig(artifact, dpi=160)
print("Chart artifact:", artifact)


## Keep the data boundary visible

Only 31 daily rows and a few payment categories entered the charting library.
A dashboard can request these summaries through the HTTP API in the same way.
Save the chart before the final cleanup if you want to retain it.


In [ ]:
assert daily_result.stats["rows"] == len(daily)
assert payments_result.stats["rows"] == len(payments)
print({"chart_rows": len(daily) + len(payments),
       "arrow_wire_bytes": daily_result.stats["wire_bytes"] + payments_result.stats["wire_bytes"]})


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
